# 40 综合评价扩展方法

> **模块**：综合评价 ｜ **数据**：`../data/textbook/14.D 宏观投资效益.txt`、`../data/textbook/14.D 医院工作质量统计指标.txt` ｜ **依赖**：numpy、scipy、scikit-learn、pandas

## 1. 问题背景

除 AHP 与 TOPSIS 外，DEA 用投入产出比评价决策单元效率，主成分法用降维合成综合指标，秩和比按秩次排序分档，三者适用于不同的数据条件。

## 2. 学习目标

- 理解 DEA 的效率值与投入产出导向
- 会用主成分构造综合评价得分
- 掌握秩和比的编秩与分档方法

## 3. 环境与数据

- 依赖：numpy、scipy、scikit-learn、pandas
- 数据：`../data/textbook/14.D 宏观投资效益.txt`、`../data/textbook/14.D 医院工作质量统计指标.txt`

## 4. 方法与实现

### 4.1 数据包络分析

**从最有利于决策单元的角度进行评价，从而避免了确定各指标在优先意义下的权重.**

- 决策单元(Decision Making Units, DMU): 每个DMU有m种投入和s种产出.
- 向量$X_j,\ Y_j$分别表示决策单元$j$的输入和输出向量; $v,\ u$分别表示输入,输出权值向量.
- 评价单元$j$的效率评价指数为: $h_j=(u^TY_j)/(v^TX_j),\ j=1,2,\cdots,n$

In [ ]:
# 例14.4问题

import numpy as np
import scipy.optimize

school_name = np.array(['A', 'B', 'C', 'D', 'E', 'F'])
X = np.array([[89.39, 64.3],
              [86.25, 99],
              [108.13, 99.6],
              [106.38, 96],
              [62.40, 96.2],
              [47.19, 79.9]])

Y = np.array([[25.2, 223],
              [28.2, 287],
              [29.4, 317],
              [26.4, 291],
              [27.2, 295],
              [25.2, 222]])

# 转化成线性规划问题, 用'01 线性规划.ipynb'的方法解
DEA = []
# 产出和投入拼起来
A_ub = np.concatenate((Y, X), axis = 1)
b_ub = np.zeros(A_ub.shape[0])
for index in range(X.shape[0]):
    # scipy的这个函数是求最小值的, 所以要取负号
    c = -np.concatenate((Y[index], np.zeros(2)))
    A_eq = [np.concatenate((np.zeros(2), X[index]))]
    b_eq = [-1]
    bounds = [(0, None), (0, None), (None, 0), (None, 0)]

    res = scipy.optimize.linprog(c=c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds)
    DEA.append(np.round(-res.fun, 5))

DEA = np.array(DEA)
DEA_effective = school_name[np.where(DEA == 1)]
print('DEA有效的学校为: ', DEA_effective)

### 4.2 主成分分析法

In [ ]:
# 例14.6
# 结果和书上有些不吻合.

import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA

data = np.loadtxt('../data/textbook/14.D 宏观投资效益.txt')
data = np.divide(np.subtract(data, data.mean(axis=0)), data.std(axis=0, ddof=1))
# print(data)
# 也可以直接用sklearn的预处理函数, 但这里ddof=1
# data = preprocessing.scale(data)

'''
# 直接使用numpy处理的
# 计算关联系数矩阵
# np.cov中的bias=False表示除以N-1而不是N
R = np.cov(data.T, bias=False)
# print(R)
eigenvalues, eigenvectors = np.linalg.eig(R)
order = eigenvalues.argsort()[::-1]
eigenvalues_order = eigenvalues[order]
eigenvectors = eigenvectors[order]
print(eigenvalues)
print(eigenvectors)
# 信息贡献率
b = np.array([eigenvalues[i]/eigenvalues.sum() for i in range(eigenvalues.shape[0])])
print(b)
# 累积贡献率
alpha = np.array([b[:i+1].sum() for i in range(b.shape[0])])
print(alpha)
'''

# PCA训练器
pca = PCA(n_components=3, copy=True, whiten=False)
# 训练并降维
newX = pca.fit_transform(data)
print(newX)
# 各个特征的方差百分比
print(pca.explained_variance_ratio_)
scores = np.matmul(newX, pca.explained_variance_ratio_)
print(np.sort(scores)[::-1])
print(np.matmul(np.eye(3), pca.explained_variance_ratio_))

### 4.3 秩和比综合评价法

In [ ]:
# 编秩函数
def get_rank(data, columns, ascending=True, R=np.zeros(1)):
    if data.ndim == 1:
        tempdata = np.array(data)[:,None]
    if not R.any():
        R = np.zeros(tempdata.shape)
    for i in columns:
        arg = tempdata[:,i].argsort(axis=0)
        if not ascending:
            arg = arg[::-1]
        begin, end = 0, 0
        # 找从begin开始相同项, 用end标记最后一个相同项的下一个
        while begin < len(arg):
            while end < len(arg) and tempdata[arg[end]][i] == tempdata[arg[begin]][i]:
                end += 1
            for j in range(begin, end):
                R[arg[j]][i] = (begin + end + 1) / 2
            begin = end
    return R.reshape(data.shape)

In [ ]:
# 例14.7

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn import linear_model

data = np.loadtxt('../data/textbook/14.D 医院工作质量统计指标.txt')
w = data[-1]
data = data[:-1]

# 编秩
R = np.zeros(data.shape)
# 效益型指标编秩
R = get_rank(data, columns=[1,5], ascending=False, R=R)
# 效益型指标编秩
R = get_rank(data, columns=[0,2,3,4], ascending=True, R=R)

# 求加权秩和比
WRSR = np.matmul(R, w)/R.shape[0]

# R_WRSR列名
R_WRSR_columns = ['x' + str(i+1) for i in range(R.shape[1])] + ['WRSR']
# R_WRSR行名
R_WRSR_index = [str(i) for i in range(1983, 1993)]
# 构建一个DataFrame. 课本表14.19
R_WRSR = pd.DataFrame(np.c_[R, WRSR], columns=R_WRSR_columns, index=R_WRSR_index)
R_WRSR_sorted = R_WRSR.sort_values(by='WRSR', ascending=True)
print(R_WRSR_sorted, '\n')

# f列名
f_columns = ['f', 'cf', 'p', 'Probit', 'WRSRfit', '排序']
f = pd.DataFrame(np.c_[np.ones((R_WRSR_sorted.shape[0], 1)), 
                       [i for i in range(1, 11)],
                       np.zeros((R_WRSR_sorted.shape[0], len(f_columns)-2))],
                 index=R_WRSR_sorted.index, columns=f_columns)
# 计算累积频率
f['p'] = f['cf'] / R.shape[0]
# 对最后一个累积频率修正
f['p'].iloc[-1] = 1 - 1/(4*R.shape[0])
# 计算概率单位, 按正态分布
f['Probit'] = norm.isf(1-f['p'], 0, 1)+5
# 以Probit为自变量, RSR/WRSR为因变量, 计算回归方程
reg = linear_model.LinearRegression()
reg.fit(np.array(f['Probit']).reshape((10, 1)), R_WRSR_sorted['WRSR'])
# 得到reg.coef_是系数, reg.intercept_是常数项. 以此计算f['WRSRfit']
f['WRSRfit'] = f['Probit'] * reg.coef_[0] + reg.intercept_
f['排序'] = np.array(f['WRSRfit'].argsort())[::-1] + 1
print(f)

## 5. 小结

方法选择取决于指标性质与样本量；不同方法的排序结果应相互对照，而非只报单一结论。

> 内容来源：教材参考实现《数学建模算法与应用》练习稿（MIT，Zhanghan Wang），原始文件已并入本篇，见 `THIRD-PARTY.md`。